### This is an experiment for new image-text retrieval web ###

**SETUP**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translator_name = 'VietAI/envit5-translation'
translate_tokenizer = AutoTokenizer.from_pretrained(translator_name)
translate_model = AutoModelForSeq2SeqLM.from_pretrained(translator_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

**DATASET LOAD**

In [7]:
# device = 'cuda' if torch.cuda.is_available else 'cpu'
device = 'cpu'
query = 'Một cảnh được quay trên 1 sân bóng đá. Trong cảnh có 2 cầu thủ mặc áo màu đỏ, 1 người có số 17, người còn lại có số 9'
output = translate_model.generate(translate_tokenizer(query, return_tensors='pt', padding=True).input_ids.to(device), max_length=1024)
translated_query = translate_tokenizer.batch_decode(output, skip_special_tokens=True)
print(translated_query)

['en: A scene was shot on a football field. There were two players in red shirts, one with the number 17, the other with the number 9.']


In [10]:
print(dataset[0].size())

torch.Size([1, 3, 378, 378])


**BATCHING DATASET**

In [15]:
BATCH_SIZE = 64
batches = []
for i in range(0, len(dataset), BATCH_SIZE):
    print(f'Batch number {i}')
    batch = dataset[i:i + BATCH_SIZE] if i < len(dataset) else dataset[i:len(dataset)]
    print(len(batch))

    batch = torch.cat(batch, dim=0)
    print(batch.size())
    batches.append(batch)

Batch number 0
64
torch.Size([64, 3, 378, 378])
Batch number 64
64
torch.Size([64, 3, 378, 378])
Batch number 128
64
torch.Size([64, 3, 378, 378])
Batch number 192
64
torch.Size([64, 3, 378, 378])
Batch number 256
16
torch.Size([16, 3, 378, 378])


**IMAGE FEATURE COMPUTATION**

In [16]:
image_batch = batches[0]

with torch.no_grad():
    image_batch_feature = model.encode_image(image_batch)
    image_batch_feature = F.normalize(image_batch_feature, dim=-1)

KeyboardInterrupt: 

**FEATURE SAVE**

In [ ]:
print(image_batch_feature.numpy().squeeze().tolist())

[0.014729292131960392, 0.058964744210243225, 0.006791392341256142, -0.007834039628505707, -0.020993584766983986, 0.010029853321611881, 0.018356427550315857, 0.0030675947200506926, 0.004231608007103205, 0.0005884041311219335, 0.0014744040090590715, -0.0372413732111454, 0.06408730149269104, -0.020269598811864853, 0.01885323040187359, 0.030054576694965363, -0.042396675795316696, -0.049855560064315796, -0.056905217468738556, -0.007575796451419592, -0.016925131902098656, 0.013709841296076775, 0.006885220762342215, 0.014025048352777958, 0.010977006517350674, -0.018436500802636147, 0.048529356718063354, 0.02161678671836853, -0.002363749546930194, 0.015650799497961998, 0.0502336360514164, 0.02342243678867817, -0.011897931806743145, 0.02295546978712082, 0.016092689707875252, 0.01353730820119381, -0.0074703809805214405, 0.03814816102385521, 0.02339242212474346, -0.018667403608560562, -0.0002478227252140641, -0.014734813943505287, 0.04005677253007889, 0.01010474469512701, -0.0284882802516222, 0.0

**DEBUGGING**

In [8]:
print(image_batch_feature.size())

torch.Size([1, 1024])
